# Setup

In [ ]:
%pip install mistralai pandas numpy

In [ ]:
from mistralai.client import Mistral
import os
import json
import time

# Initialize the Mistral client
api_key = "YOUR_API_KEY_HERE"
API_KEY = os.environ.get("MISTRAL_API_KEY", api_key)
client = Mistral(api_key=API_KEY)

# Self-Refinement

In [ ]:
"""
Self-refinement is a technique where the model critiques and improves
its own outputs through iterative feedback loops.
"""

def basic_prompt(question, model="mistral-small-latest"):

    messages = [{"role": "user", "content": question}]

    response = client.chat.complete(model=model, messages=messages)

    return response.choices[0].message.content

def critique_response(original_prompt, response, model="mistral-small-latest"):
    """
    Have the model critique its own response

    Args:
        original_prompt (str): The original prompt
        response (str): The response to critique
        model (str): Model to use

    Returns:
        str: Critique of the response
    """
    critique_prompt = f"""You previously answered the following question:

Question: {original_prompt}

Your answer was:
{response}

Please critique this answer. Identify any issues with:
1. Accuracy
2. Completeness
3. Clarity
4. Structure

Provide specific suggestions for improvement."""
# This "critique_prompt" is for demo usage, please design your prompt.

    return basic_prompt(critique_prompt, model)

def refine_response(original_prompt, response, critique, model="mistral-small-latest"):
    """
    Refine the response based on the critique

    Args:
        original_prompt (str): The original prompt
        response (str): The original response
        critique (str): The critique
        model (str): Model to use

    Returns:
        str: Refined response
    """
    refine_prompt = f"""You previously answered this question:

Question: {original_prompt}

Your original answer:
{response}

You then provided this critique:
{critique}

Now, write an improved version of your answer that addresses the issues identified in the critique."""
# This "refine_prompt" is for demo usage, please design your prompt.

    return basic_prompt(refine_prompt, model)

def self_refinement_pipeline(prompt, num_iterations=1, model="mistral-small-latest"):
    """
    Complete self-refinement pipeline

    Args:
        prompt (str): Original question/prompt
        num_iterations (int): Number of refinement iterations
        model (str): Model to use

    Returns:
        dict: Dictionary with all versions and critiques
    """
    results = {
        "original_prompt": prompt,
        "iterations": []
    }

    # Generate initial response
    print(f"Generating initial response...")
    current_response = basic_prompt(prompt, model)

    for i in range(num_iterations):
        print(f"\nIteration {i+1}/{num_iterations}")
        print("-" * 40)

        # Critique the current response
        print("Critiquing response...")
        critique = critique_response(prompt, current_response, model)

        # Refine based on critique
        print("Refining response...")
        refined_response = refine_response(prompt, current_response, critique, model)

        results["iterations"].append({
            "iteration": i + 1,
            "response": current_response,
            "critique": critique,
            "refined_response": refined_response
        })

        current_response = refined_response

    results["final_response"] = current_response
    return results

# Example usage
print("\n--- Self-Refinement Example ---\n")

prompt = "Explain quantum computing to a high school student."
print(f"Prompt: {prompt}\n")

results = self_refinement_pipeline(prompt, num_iterations=1)

print("\n" + "-" * 80)
print("INITIAL RESPONSE:")
print(results["iterations"][0]["response"])

print("\n" + "-" * 80)
print("CRITIQUE:")
print(results["iterations"][0]["critique"])

print("\n" + "-" * 80)
print("REFINED RESPONSE:")
print(results["final_response"])
